Algoritmo AdaBoost

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generar dataset de ejemplo
X, y = make_classification(n_samples=100, n_features=1, n_informative=1, 
                           n_redundant=0, n_clusters_per_class=1, random_state=42)

# Cambiar las etiquetas a {-1, 1}
y = np.where(y == 0, -1, 1)

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cantidad de clasificadores débiles
M = 10

# Inicializar pesos
n_samples = X_train.shape[0]
w = np.ones(n_samples) / n_samples

# Guardar clasificadores y sus alphas
classifiers = []
alphas = []

for m in range(M):
    # Entrenar clasificador débil
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_train, y_train, sample_weight=w)
    
    # Predecir
    y_pred = stump.predict(X_train)
    
    # Calcular error ponderado
    err_m = np.sum(w * (y_pred != y_train)) / np.sum(w)
    
    # Evitar errores muy altos
    if err_m > 0.5:
        break
    
    # Calcular alpha
    alpha_m = 0.5 * np.log((1 - err_m) / (err_m + 1e-10))
    
    # Actualizar pesos
    w = w * np.exp(-alpha_m * y_train * y_pred)
    
    # Normalizar pesos
    w /= np.sum(w)
    
    # Guardar clasificador y alpha
    classifiers.append(stump)
    alphas.append(alpha_m)
    
    print(f"Clasificador {m+1}: error = {err_m:.3f}, alpha = {alpha_m:.3f}")

# Clasificador final (predicción en test)
final_pred = np.zeros(X_test.shape[0])

for alpha_m, stump in zip(alphas, classifiers):
    final_pred += alpha_m * stump.predict(X_test)

# Convertir a -1 o 1
final_pred = np.sign(final_pred)

# Calcular accuracy
accuracy = accuracy_score(y_test, final_pred)
print(f"\nPrecisión final: {accuracy:.2f}")
